#### mport Required Libraries


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "payments", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
display(df_silver.limit(20))

In [0]:
df_gold = df_silver.select("payment_id","payment_method","payment_status","amount","payment_timestamp")
df_gold.show()

In [0]:

if not (spark.catalog.tableExists(f"{catalog}.{gold_schema}.fact_{data_source}")):
    df_gold.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_{data_source}"
    )
    print("sucessfully writed  data to delta location")
else:
    print("Doing upsert operation")
    delta_table = DeltaTable.forName(
        spark,
        f"{catalog}.{gold_schema}.fact_{data_source}"
    )

    delta_table.alias("target").merge(
        source=df_gold.alias("source"),
        condition="""
            target.payment_id = source.payment_id
        """
    ).whenMatchedUpdate(
        condition="""
        NOT (target.payment_method <=> source.payment_method)
        OR NOT (target.payment_status <=> source.payment_status)
        OR NOT (target.amount <=> source.amount)
        OR NOT (target.payment_timestamp <=> source.payment_timestamp)
    """,
        set={
             "payment_method": "coalesce(source.payment_method, target.payment_method)",
            "payment_status": "coalesce(source.payment_status, target.payment_status)",
            "amount": "coalesce(source.amount, target.amount)",
            "payment_timestamp": "coalesce(source.payment_timestamp, target.payment_timestamp)"
            
        }
    ).whenNotMatchedInsert(
        values={
            "payment_id": "source.payment_id",
            "payment_method": "source.payment_method",
            "payment_status": "source.payment_status",
            "amount": "source.amount",
            "payment_timestamp": "source.payment_timestamp"
        }
    ).execute()

